# Replay native ortholog transfer, projection, scorecards and missingness


Verify every pinned input/output, the exact original native orthogroup aggregation and receiver-training-only map, and matched scorecards/baselines. Both endpoints are prediction-grade; biological accuracy and stage/context equivalence remain unresolved. Initial artifact preflight refused missing required evaluation fields; those were corrected before any artifact directory was created.

In [1]:
import json, hashlib, ast
from pathlib import Path
import pandas as pd
from starplast import artifacts as A, transfer_records as T, organisms as O, strategies as S, strategy_catalog as N, record_scorecards as R, baselines as B
from starplast.splits import read_split
out=Path('results/ortholog_transfer_pilot_2026_10_08')
manifest=json.loads((out/'manifest.json').read_text())
sha=lambda path:hashlib.sha256(Path(path).read_bytes()).hexdigest()
for path,digest in manifest['input_sha256'].items():
    assert sha(path)==digest,path
for path,digest in manifest['outputs'].items():
    assert sha(out/path)==digest,path
split,exclusions=read_split(out/'split.json')
raw=json.loads((out/'held_out/manifest.json').read_text())
spec=A.spec_from_dict(raw['spec'])
artifact=A.read_artifact(out/'held_out',expected=spec,split=split)
assert spec.output.kind=='label_calls' and spec.confidence_kind=='none'
state=artifact.payloads['model_state.json']
projection=artifact.payloads['projection.json']
assert hashlib.sha256(A.canonical_object(projection).encode()).hexdigest()==state['source']['mapping_sha256']
ids=tuple(row.entity for row in split.assignments)
receiver=pd.read_parquet(O.nodes_path(O.TOXOPLASMA)).set_index('gene_id',drop=False)
donor=pd.read_parquet(O.nodes_path(O.FALCIPARUM))
column=state['source']['column']
other=S.Context(donor[['gene_id','orthogroup',column]],graph={},organism=O.FALCIPARUM)
context=S.Context(receiver.loc[list(ids),['gene_id','orthogroup']].reset_index(drop=True),graph={},other=other,organism=O.TOXOPLASMA)
mapped,numeric=N._through_orthologs(context,column)
mapped.index=list(ids)
assert mapped.astype(object).where(mapped.notna(),None).tolist()==projection['source_values']
labels=pd.Series(state['training_labels'],index=state['fit_entities'])
native=N._transfer_map(mapped.loc[labels.index],labels,numeric,False)(mapped).where(mapped.notna()).loc[list(split.entities('test'))]
batch=T.ortholog_transfer(mapped,labels,split=split,exclusions=exclusions,source=T.TransferSource(**state['source']),numeric_source=numeric)
assert batch.rows.prediction.tolist()==native.astype(object).where(native.notna(),None).tolist()
actual=batch.rows.astype(object).where(batch.rows.notna(),None).to_dict('records')
assert actual==[{key:row[key] for key in batch.rows.columns} for row in artifact.payloads['rows.json']]
assert batch.model_state==state and batch.class_scores.shape==(560,0)
rows=pd.DataFrame(artifact.payloads['rows.json'],columns=artifact.payloads['row_columns.json'])
scope_data=dict(artifact.payloads['card.json']['scope']); scope_data['gaps']=tuple(scope_data['gaps'])
scope=R.RecordScope(**scope_data)
normalize=lambda value:json.loads(json.dumps(value,allow_nan=False))
assert normalize(R.aggregate(rows,scope))==artifact.payloads['card.json']
assert normalize(R.class_cards(rows,scope))==artifact.payloads['class_cards.json']
baseline=B.label_baselines(labels,split,seed=split.seed)
for column in ('majority','prevalence_call'):
    baseline_rows=rows.drop(columns=['prediction','correct','abstained']).assign(prediction=baseline[column].to_numpy())
    assert normalize(R.aggregate(baseline_rows,scope))==artifact.payloads['baseline_cards.json'][column]
for path in ('starplast/transfer_records.py','scripts/freeze_transfer_pilot.py','starplast/artifacts.py'):
    ast.parse(Path(path).read_text(),feature_version=(3,10))
{'rows':len(rows),'native_calls_projection_cards':'exact','class_scores':'native unavailable','biological_admission':False,'counts':artifact.payloads['card.json']['counts']}

{'rows': 560, 'native_calls_projection_cards': 'exact', 'class_scores': 'native unavailable', 'biological_admission': False, 'counts': {'abstained': 437, 'answered': 123, 'correct': 65, 'eligible': 560, 'unique_biological_entities': 560, 'wrong': 58}}